In [73]:
import numpy as np
from scipy import stats

# Фіксація seed за номером варіанта (1)
np.random.seed(1)
daily_temps = np.random.normal(loc=20.0, scale=2.5, size=30)

mean_val = daily_temps.mean()
std_val = daily_temps.std(ddof=1)  # вибіркове стандартне відхилення (оцінка σ)
n = len(daily_temps)

print("Згенерована вибірка денних температур:\n", daily_temps)
print(f"\nВибіркове середнє (mean): {mean_val:.4f}")
print(f"Вибіркове стандартне відхилення (s): {std_val:.4f}")
print(f"Обсяг вибірки (n): {n}")

Згенерована вибірка денних температур:
 [24.06086341 18.47060897 18.67957062 17.31757844 22.16351907 14.24615326
 24.36202941 18.09698275 20.79759774 19.37657406 23.65526984 14.84964823
 19.19395699 19.03986411 22.83442361 17.25027183 19.56892948 17.80535396
 20.10553437 21.45703803 17.24845206 22.86180927 22.2539768  21.25623585
 22.25213987 18.29068035 19.69277444 17.66057641 19.3302798  21.32588867]

Вибіркове середнє (mean): 19.8502
Вибіркове стандартне відхилення (s): 2.5659
Обсяг вибірки (n): 30


In [74]:
# 1. Стандартна похибка (Standard Error - se)
se = std_val / np.sqrt(n)

# 2. Побудова 95% довірчого інтервалу через t-розподіл
ci_mean_95 = stats.t.interval(0.95, df=n - 1, loc=mean_val, scale=se)

print(f"Стандартна похибка (se): {se:.4f}")
print(f"95% довірчий інтервал для середньої температури: {ci_mean_95}")

Стандартна похибка (se): 0.4685
95% довірчий інтервал для середньої температури: (np.float64(18.892025108565047), np.float64(20.808280338156845))


In [75]:
# 1. Частка днів із температурою вище липневого значення (20°C)
threshold = 20.0
p_hat = (daily_temps > threshold).mean()

# 2. Перевірка умови нормального наближення
n_p = n * p_hat
n_1p = n * (1 - p_hat)

print(f"Вибіркова частка (p_hat): {p_hat:.4f}")
print(f"n * p_hat = {n_p:.2f} (має бути >= 5)")
print(f"n * (1 - p_hat) = {n_1p:.2f} (має бути >= 5)")

# 3. Побудова 95% довірчого інтервалу для частки
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(0.95, loc=p_hat, scale=se_p)

print(f"95% довірчий інтервал для частки: {ci_prop_95}")

Вибіркова частка (p_hat): 0.4333
n * p_hat = 13.00 (має бути >= 5)
n * (1 - p_hat) = 17.00 (має бути >= 5)
95% довірчий інтервал для частки: (np.float64(0.2560114457146738), np.float64(0.6106552209519929))


In [76]:
levels = [0.90, 0.95, 0.99]
results = []

for alpha in levels:
    ci = stats.t.interval(alpha, df=n - 1, loc=mean_val, scale=se)
    width = ci[1] - ci[0]
    results.append({
        "Рівень довіри": f"{int(alpha * 100)}%",
        "Нижня межа": round(ci[0], 4),
        "Верхня межа": round(ci[1], 4),
        "Ширина інтервалу": round(width, 4)
    })

import pandas as pd
df_ci_widths = pd.DataFrame(results)
display(df_ci_widths)

,Рівень довіри,Нижня межа,Верхня межа,Ширина інтервалу
0,90%,19.0542,20.6461,1.5920
1,95%,18.8920,20.8083,1.9163
2,99%,18.5589,21.1414,2.5826


### Відповіді на контрольні запитання:

1. **Різниця між точковою оцінкою і довірчим інтервалом:**
   - Точкова оцінка (наприклад, вибіркове середнє $\bar{x}$) дає лише одне конкретне число, яке майже ніколи не дорівнює істинному параметру генеральної сукупності через випадкову похибку вибірки. Довірчий інтервал охоплює діапазон правдоподібних значень із заданим рівнем надійності, що дозволяє оцінити точність і ступінь невизначеності нашої оцінки.

2. **Чому для довірчого інтервалу середнього при невідомому $\sigma$ використовують $t$-розподіл:**
   - Коли справжнє стандартне відхилення генеральної сукупності $\sigma$ невідоме, ми змушені замінювати його вибірковим оцінювачем $s$. У разі малих або помірних вибірок це додає додаткову невизначеність, яку ідеально компенсує $t$-розподіл Стьюдента (його хвости ширші за нормальний розподіл).

3. **Що насправді означає "95% довіри":**
   - Це частотна інтерпретація: якщо повторити експеримент та побудову інтервалу багато разів, у 95% випадків побудовані інтервали покриють істинне значення параметра. Помилковим є твердження про «95% ймовірності для одного конкретного інтервалу», оскільки межі нашої єдиної вибірки вже зафіксовані, і параметр або належить їм (з імовірністю 1), або ні (0).

4. **Три фактори, що визначають ширину довірчого інтервалу:**
   - **Рівень довіри ($1 - \alpha$):** вищий рівень довіри робить інтервал ширшим.
   - **Вибіркове стандартне відхилення ($s$):** більший розкид у даних робить інтервал ширшим.
   - **Обсяг вибірки ($n$):** більша вибірка робить інтервал вужчим через зменшення стандартної похибки.